In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap, workshop_retrieved_references

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


## Estratégias avançadas de chunking fornecidas pelo Amazon Bedrock Knowledge Bases

Neste notebook, criaremos 3 knowledge bases para fornecer código de exemplo para as seguintes opções de chunking suportadas pelo Amazon Bedrock Knowledge Bases: 
1. Fixed chunking
2. Semantic chunking
3. Hierarchical chunking
4. Custom chunking usando Lambda function


O chunking divide o texto em segmentos menores antes do embedding. A estratégia de chunking não pode ser modificada depois que você cria a data source.
Atualmente, o Amazon Bedrock Knowledge Bases suporta apenas algumas opções de chunking integradas: sem chunking, chunking de tamanho fixo e chunking padrão. 

* Com as funcionalidades de Semantic e Hierarchical chunking (além das opções existentes), os clientes podem ter mais controle sobre como seus dados são processados e divididos em chunks usando Lambda function.


Usaremos um relatório 10K sintético como dados para uma empresa fictícia chamada `Octank Financial` para demonstrar a solução.
Após criar as knowledge bases, avaliaremos os resultados no mesmo dataset. O foco será melhorar a qualidade dos resultados de busca, o que por sua vez melhorará a precisão das respostas geradas pelo foundation model. 

## 1. Importar as bibliotecas necessárias
O primeiro passo é instalar os pacotes de pré-requisitos.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
%pip install ragas==0.1.9 --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import botocore
botocore.__version__

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import time

# Get the current timestamp
current_time = time.time()

# Format the timestamp as a string
timestamp_str = time.strftime("%Y%m%d%H%M%S", time.localtime(current_time))[-7:]
# Create the suffix using the timestamp
suffix = f"{timestamp_str}"
knowledge_base_name_standard = 'standard-kb'
knowledge_base_name_hierarchical = 'hierarchical-kb'
knowledge_base_name_semantic = 'semantic-kb'
knowledge_base_name_custom = 'custom-chunking-kb'
knowledge_base_description = "Knowledge Base containing complex PDF."
bucket_name = f'{knowledge_base_name_standard}-{suffix}'
intermediate_bucket_name = f'{knowledge_base_name_standard}-intermediate-{suffix}'
lambda_function_name = f'{knowledge_base_name_custom}-lambda-{suffix}'
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

data_source=[{"type": "S3", "bucket_name": bucket_name}]

## 2 - Criar knowledge bases com estratégia de fixed chunking
Vamos começar criando uma [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) para armazenar os menus do restaurante. As Knowledge Bases permitem integração com diferentes bancos de dados vetoriais, incluindo [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() e [MongoDB Atlas](). Para este exemplo, integraremos a knowledge base com o Amazon OpenSearch Serverless. Para isso, usaremos a classe auxiliar `BedrockKnowledgeBase` que criará a knowledge base e todos os seus pré-requisitos:
1. IAM roles e policies
2. Bucket S3
3. Políticas de encryption, network e data access do Amazon OpenSearch Serverless
4. Collection do Amazon OpenSearch Serverless
5. Índice vetorial do Amazon OpenSearch Serverless
6. Knowledge base
7. Data source da Knowledge base

Primeiro criaremos uma knowledge base usando a estratégia de fixed chunking seguida pela estratégia de hierarchical chunking. 

Valores dos parâmetros: 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_standard = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_standard}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source,
    chunking_strategy = "FIXED_SIZE", 
    suffix = f'{suffix}-f'
)

## 2.1 Fazer upload do dataset para o Amazon S3
Agora que criamos a knowledge base, vamos preenchê-la com o dataset do relatório `Octank financial 10K`. A data source da Knowledge Base espera que os dados estejam disponíveis no bucket S3 conectado a ela e as alterações nos dados podem ser sincronizadas com a knowledge base usando a chamada de API `StartIngestionJob`. Neste exemplo, usaremos a [abstração do boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent/client/start_ingestion_job.html) da API, por meio de nossa classe auxiliar. 

Vamos primeiro fazer upload dos dados de menus disponíveis na pasta `dataset` para o S3.

In [ ]:
import os

def upload_directory(path, bucket_name):
    for root, dirs, files in os.walk(path):
        for file in files:
            file_to_upload = os.path.join(root, file)
            if file not in ["LICENSE", "NOTICE", "README.md"]:
                print(f"uploading file {file_to_upload} to {bucket_name}")
                s3_client.upload_file(file_to_upload, bucket_name, file)
            else:
                print(f"Skipping file {file_to_upload}")

upload_directory("../synthetic_dataset", bucket_name)


Agora iniciamos o ingestion job.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_standard.start_ingestion_job()

Por fim, salvamos o Knowledge Base Id para testar a solução em uma etapa posterior. 

In [ ]:
kb_id_standard = knowledge_base_standard.get_knowledge_base_id()

### 2.2 Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). 

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

A resposta correta para essa query, de acordo com o par de QA ground truth, é: 

```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow. 
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
query = "Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019."

In [ ]:
time.sleep(20)
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_standard,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

Como você pode ver, com a API retrieve and generate obtemos a resposta final diretamente. Agora vamos observar as citações da API `RetreiveAndGenerate`. Uma vez que nosso foco principal neste notebook é observar os chunks recuperados e as citações retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, ele provavelmente gerará uma resposta de alta qualidade. 

In [ ]:
def citations_rag_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret,1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

In [ ]:
response_standard = workshop_retrieved_references(response)
print("# of citations or chunks used to generate the response: ", len(response_standard))
citations_rag_print(response_standard)

Vamos agora inspecionar as informações de origem da knowledge base com a API retrieve.

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
def response_print(response_ret):
#structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
    for num,chunk in enumerate(response_ret['retrievalResults'],1):
        print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
        print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
        print(f'Chunk {num} Score: ',chunk['score'],end='\n'*2)
        print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)


In [ ]:
response_standard_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_standard, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)

print("# of retrieved results: ", len(response_standard_ret['retrievalResults']))
response_print(response_standard_ret)

Como você pode notar, com `fixed chunking` obtemos 5 resultados recuperados conforme solicitado na API usando `semantic similarity`, que é o padrão para a `Retrieve API`. Vamos agora usar a estratégia de `hierarchical chunking` e inspecionar os resultados recuperados usando a API `RetrieveAndGenerate` bem como a API `Retrieve`. 

## 3. Criar knowledge bases com estratégia de hierarchical chunking

**Conceito**

Hierarchical chunking: organiza seus dados em uma estrutura hierárquica, permitindo uma recuperação mais granular e eficiente com base nos relacionamentos inerentes aos seus dados. Organizar seus dados em uma estrutura hierárquica permite que seu workflow de RAG navegue e recupere informações de forma eficiente em datasets complexos e aninhados.
Após os documentos serem analisados, o primeiro passo é dividir os documentos com base no tamanho de chunking parent e child. Os chunks são então organizados em uma estrutura hierárquica, onde o parent chunk (nível superior) representa chunks maiores (por exemplo, documentos ou seções), e os child chunks (nível inferior) representam chunks menores (por exemplo, parágrafos ou sentenças). O relacionamento entre os parent e child chunks é mantido. Essa estrutura hierárquica permite a recuperação e navegação eficientes do corpus.

**Benefícios:**

- Recuperação eficiente: a estrutura hierárquica permite uma recuperação mais rápida e direcionada de informações relevantes; primeiro realizando a busca semântica no child chunk e depois retornando o parent chunk durante a recuperação. Ao substituir os children chunks pelo parent chunk, fornecemos um contexto amplo e abrangente ao FM.
- Preservação de contexto: organizar o corpus de forma hierárquica ajuda a preservar os relacionamentos contextuais entre chunks, o que pode ser benéfico para gerar texto coerente e contextualmente relevante.

><br>          
>Nota:
>No hierarchical chunking, parent chunks são retornados e a busca é realizada nos children chunks, portanto, você pode ver menos resultados de busca retornados, pois um parent pode ter múltiplos children.
> <br></br>       

O hierarchical chunking é mais adequado para documentos complexos que possuem uma estrutura aninhada ou hierárquica, como manuais técnicos, documentos legais ou artigos acadêmicos com formatação complexa e tabelas aninhadas.

**Valores dos parâmetros:** 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_hierarchical = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_hierarchical}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source,
    chunking_strategy = "HIERARCHICAL", 
    suffix = f'{suffix}-h'
)

Agora inicie o ingestion job. Como estamos usando os mesmos documentos usados para fixed chunking, estamos pulando a etapa de upload de documentos para o bucket S3. 

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_hierarchical.start_ingestion_job()

Salve o knowledge base id para testes posteriores. 

In [ ]:
kb_id_hierarchical = knowledge_base_hierarchical.get_knowledge_base_id()

### 3.1 Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). 

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

A resposta correta para essa query, de acordo com o par de QA ground truth, é: 

```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow. 
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
time.sleep(20)
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_hierarchical,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

Como você pode ver, com a API `RetreiveAndGenerate` obtemos a resposta final diretamente. Agora vamos observar as citações da API `RetreiveAndGenerate`. Uma vez que nosso foco principal neste notebook é observar os chunks recuperados e as citações retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, ele provavelmente gerará uma resposta de alta qualidade. 

In [ ]:
response_hierarchical = workshop_retrieved_references(response)
print("# of citations or chunks used to generate the response: ", len(response_hierarchical))
citations_rag_print(response_hierarchical)

Vamos agora recuperar as informações de origem da knowledge base com a API retrieve.

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
response_hierarchical_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_hierarchical, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)

print("# of retrieved results: ", len(response_hierarchical_ret['retrievalResults']))
response_print(response_hierarchical_ret)

"><br>
> Nota:
>Como você pode ver na resposta acima, a API `retrieve` retornou apenas 3 resultados de busca ou chunks, embora 5 tenham sido passados na requisição. O motivo é que com o chunking `hierarchical`, parent chunks são retornados pela API enquanto a busca é realizada nos `children chunks` e um `parent chunk` pode ter múltiplos `children chunks`. Portanto, a resposta retornou apenas 3 chunks enquanto a busca foi realizada em 5 `children chunks`.
><br></br>

## 4. Criar knowledge bases com estratégia de semantic chunking

**Conceito**

O semantic chunking analisa os relacionamentos dentro de um texto e o divide em chunks significativos e completos, que são derivados com base na similaridade semântica calculada pelo modelo de embedding. Essa abordagem preserva a integridade das informações durante a recuperação, ajudando a garantir resultados precisos e contextualmente apropriados.
O Amazon Bedrock Knowledge Bases primeiro divide os documentos em chunks com base no tamanho de token especificado. Embeddings são criados para cada chunk, e chunks similares no espaço de embedding são combinados com base no similarity threshold e buffer size, formando novos chunks. Consequentemente, o tamanho do chunk pode variar entre os chunks.

**Benefícios**

- Ao focar no significado e contexto do texto, o semantic chunking melhora significativamente a qualidade da recuperação. Deve ser usado em cenários onde manter a integridade semântica do texto é crucial.

- Embora este método seja mais intensivo computacionalmente do que o fixed-size chunking, pode ser benéfico para dividir documentos onde os limites contextuais não são claros — por exemplo, documentos legais ou manuais técnicos.[[1]](#https://www.mongodb.com/developer/products/atlas/choosing-chunking-strategy-rag/)

**Valores dos parâmetros:**
 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_semantic = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_semantic}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source, 
    chunking_strategy = "SEMANTIC", 
    suffix = f'{suffix}-s'
)

Agora inicie o ingestion job. Como estamos usando os mesmos documentos usados para fixed chunking, estamos pulando a etapa de upload de documentos para o bucket S3. 

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_semantic.start_ingestion_job()

In [ ]:
kb_id_semantic = knowledge_base_semantic.get_knowledge_base_id()

### 4.1 Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). 

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

A resposta correta para essa query, de acordo com o par de QA ground truth, é: 

```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow. 
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
time.sleep(20)

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_semantic,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

Como você pode ver, com a API `RetreiveAndGenerate` obtemos a resposta final diretamente. Agora vamos observar as citações da API `RetreiveAndGenerate`. Uma vez que nosso foco principal neste notebook é observar os chunks recuperados e as citações retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, ele provavelmente gerará uma resposta de alta qualidade. 

In [ ]:
response_semantic = workshop_retrieved_references(response)
print("# of citations or chunks used to generate the response: ", len(response_semantic))
citations_rag_print(response_semantic)

Vamos agora recuperar as informações de origem da knowledge base com a API retrieve.

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
response_semantic_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_semantic, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)
print("# of citations or chunks used to generate the response: ", len(response_semantic_ret['retrievalResults']))
response_print(response_semantic_ret)

## 5. Opção de custom chunking usando Lambda Functions
Ao criar uma Knowledge Bases (KB) para o Amazon Bedrock, você pode conectar uma Lambda function para especificar sua lógica de chunking customizada. Durante a ingestão, se a lambda function for fornecida, a Knowledge Bases executará a lambda function e armazenará os valores de entrada e saída no bucket S3 intermediário fornecido.

> <br>
> Nota: A Lambda function com KB pode ser usada para adicionar lógica de chunking customizada, bem como para processar seus chunks, por exemplo, adicionando metadata em nível de chunk. Neste exemplo, estamos focando no uso da Lambda function para lógica de chunking customizada.
> <br></br>

### 5.1 Criar a Lambda Function

Agora criaremos uma lambda function que conterá o código para custom chunking. Para isso, vamos:

1. Criar o arquivo `lambda_function.py` que contém a lógica para custom chunking.
2. Criar a IAM role para nossa Lambda function.
3. Criar a lambda function com as permissões necessárias.

#### Criar o código da função
 Vamos criar a lambda function que implementa as funções para `ler seu arquivo do bucket intermediário`, `processar o conteúdo com lógica de chunking customizada` e `escrever a saída de volta no bucket S3`. 

In [ ]:
%%writefile lambda_function.py
import json
from abc import abstractmethod, ABC
from typing import List
from urllib.parse import urlparse
import boto3
import logging
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

class Chunker(ABC):
    @abstractmethod
    def chunk(self, text: str) -> List[str]:
        raise NotImplementedError()
        
class SimpleChunker(Chunker):
    def chunk(self, text: str) -> List[str]:
        words = text.split()
        return [' '.join(words[i:i+100]) for i in range(0, len(words), 100)]

def lambda_handler(event, context):
    logger.debug('input={}'.format(json.dumps(event)))
    s3 = boto3.client('s3')

    # Extract relevant information from the input event
    input_files = event.get('inputFiles')
    input_bucket =  event.get('bucketName')

    
    if not all([input_files, input_bucket]):
        raise ValueError("Missing required input parameters")
    
    output_files = []
    chunker = SimpleChunker()

    for input_file in input_files:
        content_batches = input_file.get('contentBatches', [])
        file_metadata = input_file.get('fileMetadata', {})
        original_file_location = input_file.get('originalFileLocation', {})

        processed_batches = []
        
        for batch in content_batches:
            input_key = batch.get('key')

            if not input_key:
                raise ValueError("Missing uri in content batch")
            
            # Read file from S3
            file_content = read_s3_file(s3, input_bucket, input_key)
            
            # Process content (chunking)
            chunked_content = process_content(file_content, chunker)
            
            output_key = f"Output/{input_key}"
            
            # Write processed content back to S3
            write_to_s3(s3, input_bucket, output_key, chunked_content)
            
            # Add processed batch information
            processed_batches.append({
                'key': output_key
            })
        
        # Prepare output file information
        output_file = {
            'originalFileLocation': original_file_location,
            'fileMetadata': file_metadata,
            'contentBatches': processed_batches
        }
        output_files.append(output_file)
    
    result = {'outputFiles': output_files}
    
    return result
    

def read_s3_file(s3_client, bucket, key):
    response = s3_client.get_object(Bucket=bucket, Key=key)
    return json.loads(response['Body'].read().decode('utf-8'))

def write_to_s3(s3_client, bucket, key, content):
    s3_client.put_object(Bucket=bucket, Key=key, Body=json.dumps(content))    

def process_content(file_content: dict, chunker: Chunker) -> dict:
    chunked_content = {
        'fileContents': []
    }
    
    for content in file_content.get('fileContents', []):
        content_body = content.get('contentBody', '')
        content_type = content.get('contentType', '')
        content_metadata = content.get('contentMetadata', {})
        
        words = content['contentBody']
        chunks = chunker.chunk(words)
        
        for chunk in chunks:
            chunked_content['fileContents'].append({
                'contentType': content_type,
                'contentMetadata': content_metadata,
                'contentBody': chunk
            })
    
    return chunked_content

Os valores padrão das estratégias de chunking fornecidos pelas knowledge bases são os seguintes: 

**Valores dos parâmetros:**
 
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

Para implementar nossa lógica customizada, incluímos uma opção na classe `knowledge_base.py` para passar o valor `CUSTOM"`. 
Se você passar a estratégia de chunking como `CUSTOM` nesta classe, ela fará o seguinte: 

1. Selecionará a `chunkingStrategy` como `NONE`. 
2. Adicionará `customTransformationConfiguration` ao `vectorIngestionConfiguration` da seguinte forma: 

```
{
...
   "vectorIngestionConfiguration": {
    "customTransformationConfiguration": { 
         "intermediateStorage": { 
            "s3Location": { 
               "uri": "string"
            }
         },
         "transformations": [
            {
               "transformationFunction": {
                  "lambdaConfiguration": {
                     "lambdaArn": "string"
                  }
               },
               "stepToApply": "string" // enum of POST_CHUNKING
            }
         ]
      },
      "chunkingConfiguration": {
         "chunkingStrategy": "NONE"
         ...
   }
}

```

In [ ]:
knowledge_base_custom = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name_custom}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source,
    lambda_function_name=lambda_function_name,
    intermediate_bucket_name=intermediate_bucket_name, 
    chunking_strategy = "CUSTOM", 
    suffix = f'{suffix}-c'
)

Agora inicie o ingestion job. 

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_custom.start_ingestion_job()

In [ ]:
kb_id_custom = knowledge_base_custom.get_knowledge_base_id()

### 5.2 Testar a Knowledge Base
Agora que a Knowledge Base está disponível, podemos testá-la usando as funções [**retrieve**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html) e [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). 

#### Testando a Knowledge Base com a API Retrieve and Generate

Vamos primeiro testar a knowledge base usando a API retrieve and generate. Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

query = `Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019.`

A resposta correta para essa query, de acordo com o par de QA ground truth, é: 

```
The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals the following:
- Cash generated from operating activities amounted to $710 million, which can be attributed to a $700 million profit and non-cash charges such as depreciation and amortization.
- Cash outflow from investing activities totaled $240 million, with major expenditures being the acquisition of property, plant, and equipment ($200 million) and marketable securities ($60 million), partially offset by the sale of property, plant, and equipment ($40 million) and maturing marketable securities ($20 million).
- Financing activities resulted in a cash inflow of $350 million, stemming from the issuance of common stock ($200 million) and long-term debt ($300 million), while common stock repurchases ($50 million) and long-term debt payments ($100 million) reduced the cash inflow. 
Overall, Octank Financial experienced a net cash enhancement of $120 million in 2019, bringing their total cash and cash equivalents to $210 million.
```

In [ ]:
time.sleep(10)

response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_custom,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

print(response['output']['text'],end='\n'*2)

Como você pode ver, com a API `RetreiveAndGenerate` obtemos a resposta final diretamente. Agora vamos observar as citações da API `RetreiveAndGenerate`. Uma vez que nosso foco principal neste notebook é observar os chunks recuperados e as citações retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, ele provavelmente gerará uma resposta de alta qualidade. 

In [ ]:
response_custom = workshop_retrieved_references(response)
print("# of citations or chunks used to generate the response: ", len(response_custom))
citations_rag_print(response_custom)

Vamos agora recuperar as informações de origem da knowledge base com a API retrieve.

#### Testando a Knowledge Base com a API Retrieve
Se você precisa de uma camada extra de controle, pode recuperar os chunks que melhor correspondem à sua query usando a API retrieve. Nesta configuração, podemos definir o número desejado de resultados e controlar a resposta final com a lógica da sua própria aplicação. A API então fornece o conteúdo correspondente, sua localização no S3, o similarity score e o metadata do chunk.

In [ ]:
response_custom_ret = bedrock_agent_runtime_client.retrieve(
    knowledgeBaseId=kb_id_custom, 
    nextToken='string',
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults":5,
        } 
    },
    retrievalQuery={
        'text': query
    }
)
print("# of citations or chunks used to generate the response: ", len(response_custom_ret['retrievalResults']))
response_print(response_custom_ret)

Em todos os casos, ao avaliar uma query, obtivemos a resposta correta. No entanto, quando você está construindo uma aplicação RAG, precisa avaliar com um grande número de Perguntas e Respostas para identificar melhorias de precisão. No próximo passo, usaremos o framework open source RAG Assessment (RAGAS) para avaliar as respostas no `seu dataset` para as métricas relacionadas à avaliação da qualidade do contexto ou dos resultados de busca.
Vamos focar apenas em 2 métricas: 

1. Context recall
2. Context relevancy

## 6. Avaliando resultados de busca usando o framework RAG Assessment (RAGAS) no seu dataset
Você pode usar o framework RAGAS para avaliar seus resultados para cada estratégia de chunking. Essa abordagem pode ajudá-lo a fornecer orientação factual sobre qual estratégia de chunking usar para o seu dataset. 

Idealmente, você deve considerar otimizar outros parâmetros também, por exemplo, no caso do hierarchical chunking, você deve considerar experimentar diferentes tamanhos para o parent chunk ou child chunk. 

A abordagem abaixo fornecerá heurísticas sobre qual estratégia poderia ser usada com base nos parâmetros padrão recomendados pelo Amazon Bedrock Knowledge Bases. 

In [ ]:
print("Semantic: ", kb_id_semantic)
print("Standard: ", kb_id_standard)
print("Hierarchical: ", kb_id_hierarchical)
print("Custom chunking: ", kb_id_custom)

#### Avaliação
Nesta seção utilizaremos o RAGAS para avaliar os resultados de busca usando as seguintes métricas:
1. **Context Recall:** O Context recall mede a extensão em que o contexto recuperado se alinha com a resposta anotada, tratada como o ground truth. É calculado com base no ground truth e no contexto recuperado, e os valores variam entre 0 e 1, com valores mais altos indicando melhor desempenho.

2. **Context relevancy:** Essa métrica avalia a relevância do contexto recuperado, calculada com base tanto na pergunta quanto nos contextos. Os valores ficam dentro do intervalo (0, 1), com valores mais altos indicando melhor relevância.

In [ ]:
import json
import re
import pandas as pd

MODEL_ID_EVAL = os.getenv("BEDROCK_EVALUATION_MODEL_ID", "us.anthropic.claude-sonnet-4-6")
question = "Provide a summary of consolidated statements of cash flows of Octank Financial for the fiscal years ended December 31, 2019."
ground_truth = "The cash flow statement for Octank Financial in the year ended December 31, 2019 reveals operating cash flow of $710 million, investing cash outflow of $240 million, financing cash inflow of $350 million, and a net cash enhancement of $120 million."
evaluation_client = boto3.client("bedrock-runtime", region_name=region)

def bounded_chunking_evaluation(kb_id):
    retrieved = bedrock_agent_runtime_client.retrieve(
        knowledgeBaseId=kb_id,
        retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5}},
        retrievalQuery={"text": question},
    )
    contexts = "\n".join(
        item.get("content", {}).get("text", "")
        for item in retrieved.get("retrievalResults", [])
    )
    prompt = (
        "Evaluate retrieval quality. Return only JSON with numeric values "
        "from 0 to 1 for context_recall and context_precision.\n"
        f"Question: {question}\n"
        f"Ground truth: {ground_truth}\n"
        f"Retrieved context:\n{contexts[:12000]}"
    )
    payload = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 96,
        "temperature": 0,
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
    })
    response = evaluation_client.invoke_model(
        body=payload,
        modelId=MODEL_ID_EVAL,
        accept="application/json",
        contentType="application/json",
    )
    text = json.loads(response["body"].read())["content"][0]["text"]
    match = re.search(r"\{.*\}", text, re.DOTALL)
    scores = json.loads(match.group(0)) if match else {}
    return pd.DataFrame([{
        "context_recall": float(scores.get("context_recall", 0)),
        "context_precision": float(scores.get("context_precision", 0)),
        "evaluation_mode": "bounded_bedrock_smoke",
    }])

results_standard = bounded_chunking_evaluation(kb_id_standard)
results_heirarchical = bounded_chunking_evaluation(kb_id_hierarchical)
results_semantic = bounded_chunking_evaluation(kb_id_semantic)
results_custoom = bounded_chunking_evaluation(kb_id_custom)

In [ ]:
print("Bounded Bedrock evaluation completed for all chunking strategies.")

In [ ]:
import pandas as pd
pd.options.display.max_colwidth = 800
print("Fixed Chunking Evaluation for synthetic 10K report")
print("--------------------------------------------------------------------")
print("Average context_recall: ", results_standard["context_recall"].mean())
print("Average context_relevancy: ", results_standard["context_precision"].mean(), "\n")

print("Hierarchical Chunking Evaluation for synthetic 10K report")
print("--------------------------------------------------------------------")
print("Average context_recall: ", results_heirarchical["context_recall"].mean())
print("Average context_relevancy: ", results_heirarchical["context_precision"].mean(), "\n")

print("Semantic Chunking Evaluation for synthetic 10K report")
print("--------------------------------------------------------------------")
print("Average context_recall: ", results_semantic["context_recall"].mean())
print("Average context_relevancy: ", results_semantic["context_precision"].mean(), "\n")

print("Custom Chunking Evaluation for synthetic 10K report")
print("--------------------------------------------------------------------")
print("Average context_recall: ", results_custoom["context_recall"].mean())
print("Average context_relevancy: ", results_custoom["context_precision"].mean())


In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
